In [ ]:
import json, urllib.request, zipfile, io, subprocess, sys
from pathlib import Path
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
_demo_root = Path('.hyper-demo').resolve()
_demo_tools = _demo_root / 'tools'
_demo_packages = _demo_root / 'packages'
_tags = set(sys_tags())
with urllib.request.urlopen('https://pypi.org/pypi/uv/json') as response:
    _uv_release = json.load(response)
_uv_wheel = next(f for f in _uv_release['urls'] if f['filename'].endswith('.whl') and parse_wheel_filename(f['filename'])[3] & _tags)
with urllib.request.urlopen(_uv_wheel['url']) as response:
    _wheel_bytes = response.read()
import hashlib
assert hashlib.sha256(_wheel_bytes).hexdigest() == _uv_wheel['digests']['sha256']
zipfile.ZipFile(io.BytesIO(_wheel_bytes)).extractall(_demo_tools)
_uv = next(p for p in _demo_tools.rglob('uv') if p.is_file())
_uv.chmod(0o755)
subprocess.run([str(_uv), 'pip', 'install', '--target', str(_demo_packages), 'hyperhtml==0.1.2'], check=True)
sys.path.insert(0, str(_demo_packages))
from hyperhtml import _native

_demo_extension = _demo_root / 'hyper_demo.py'
_demo_extension.write_text('"""Load with `%load_ext hyperhtml.ipython`; run `%%hyper Button`."""\n\nfrom dataclasses import dataclass\nfrom html import escape\nfrom inspect import isawaitable, signature\nfrom keyword import iskeyword\nfrom uuid import uuid4\n\nfrom hyperhtml import _native\n\n\n@dataclass\nclass HyperPreview:\n    html: str\n    python: str\n\n    def _repr_html_(self):\n        ident = f\'hyper-{uuid4().hex}\'\n        return f\'\'\'<div id="{ident}">\n<style>\n#{ident} .panel {{display:none}}\n#{ident} input:checked + label + .panel {{display:block}}\n#{ident} {{display:grid;grid-template-columns:auto auto 1fr;gap:8px}}\n#{ident} input {{position:absolute;opacity:0;width:0}}\n#{ident} label {{grid-row:1;cursor:pointer;padding:6px 12px;border:1px solid #888;border-radius:4px}}\n#{ident} input:checked + label {{background:#ddd;color:#111}}\n#{ident} input:focus-visible + label {{outline:2px solid #268bd2}}\n#{ident} .panel {{grid-row:2;grid-column:1 / -1}}\n#{ident} pre {{overflow:auto;max-height:600px;white-space:pre}}\n</style>\n<input type="radio" name="{ident}" id="{ident}-preview" checked>\n<label for="{ident}-preview">Preview</label>\n<div class="panel"><iframe title="Hyper preview" sandbox="" style="width:100%;height:360px;border:0;background:white" srcdoc="{escape(self.html, quote=True)}"></iframe></div>\n<input type="radio" name="{ident}" id="{ident}-python">\n<label for="{ident}-python">Python</label>\n<div class="panel"><pre><code>{escape(self.python)}</code></pre></div>\n</div>\'\'\'\n\n\ndef load_ipython_extension(ipython):\n    def hyper(line, cell):\n        """Compile a named component and preview it using notebook variables."""\n        name = line.strip()\n        if not name.isidentifier() or iskeyword(name):\n            raise ValueError(\'Provide a component name, for example: %%hyper Button\')\n\n        filename = f\'{name}.hyper\'\n        python = _native.transpile(cell, filename)\n        namespace = ipython.user_ns.copy()\n        exec(compile(python, filename, \'exec\'), namespace)\n        component = namespace[name]\n\n        props = {\n            prop: ipython.user_ns[prop]\n            for prop in signature(component).parameters\n            if prop in ipython.user_ns\n        }\n        rendered = component(**props)\n        if isawaitable(rendered):\n            rendered.close()\n            raise TypeError(\'The Hyper preview requires a synchronous component\')\n\n        ipython.user_ns[name] = component\n        return HyperPreview(str(rendered), python)\n\n    ipython.register_magic_function(hyper, magic_kind=\'cell\', magic_name=\'hyper\')\n\n\ndef unload_ipython_extension(ipython):\n    ipython.magics_manager.magics[\'cell\'].pop(\'hyper\', None)\n')
sys.path.insert(0, str(_demo_root))
%reload_ext hyper_demo
from IPython.display import Javascript, display
display(Javascript('(() => {\n  const language = \'hyper\';\n  if (!monaco.languages.getLanguages().some(lang => lang.id === language)) {\n    monaco.languages.register({id: language});\n  }\n  window.hyperHighlight?.dispose();\n  window.hyperHighlight = monaco.languages.setMonarchTokensProvider(language, {\n    keywords: [\'for\', \'in\', \'if\', \'else\', \'elif\', \'end\', \'from\', \'import\', \'as\',\n      \'def\', \'return\', \'yield\', \'await\', \'async\', \'True\', \'False\', \'None\', \'and\', \'or\', \'not\'],\n    tokenizer: {\n      root: [\n        [/^%%hyper.*$/, \'metatag\'],\n        [/^\\s*---\\s*$/, \'delimiter\'],\n        [/<\\/?[\\w-]+/, {token: \'tag\', next: \'@tag\'}],\n        [/\\{/, {token: \'delimiter.bracket\', next: \'@expression\'}],\n        {include: \'@python\'},\n      ],\n      tag: [\n        [/\\s+/, \'white\'],\n        [/\\/?\\>/, {token: \'tag\', next: \'@pop\'}],\n        [/\\{/, {token: \'delimiter.bracket\', next: \'@expression\'}],\n        [/"[^"\\n]*"|\'[^\'\\n]*\'/, \'attribute.value\'],\n        [/[\\w:-]+/, \'attribute.name\'],\n        [/=/, \'delimiter\'],\n      ],\n      expression: [\n        [/\\{/, {token: \'delimiter.bracket\', next: \'@push\'}],\n        [/\\}/, {token: \'delimiter.bracket\', next: \'@pop\'}],\n        {include: \'@python\'},\n      ],\n      python: [\n        [/#.*$/, \'comment\'],\n        [/"[^"\\n]*"|\'[^\'\\n]*\'/, \'string\'],\n        [/\\b\\d+(\\.\\d+)?\\b/, \'number\'],\n        [/[a-zA-Z_]\\w*/, {cases: {\'@keywords\': \'keyword\', \'@default\': \'identifier\'}}],\n        [/[()[\\]]/, \'@brackets\'],\n        [/[=:,.+*/-]/, \'delimiter\'],\n      ],\n    },\n  });\n  // SolveIT caches magic-to-language mappings when its editor first opens.\n  if (typeof _langMap !== \'undefined\') _langMap = undefined;\n  for (const model of monaco.editor.getModels()) {\n    if (model.getValue().startsWith(\'%%hyper\')) monaco.editor.setModelLanguage(model, language);\n  }\n})();\n'))
print('Hyper ready. Run the next cell, then edit and rerun the Hyper cell.')


In [ ]:
names = ['Ada', 'Lin', 'Chris']

In [ ]:
%%hyper Greeting
names: list[str]
---
<main style="font: 20px system-ui; padding: 24px">
<h1>Hyper in SolveIT</h1>
<ul>
    for name in names:
        <li>Hello, {name}!</li>
    end
</ul>
</main>